# Momentum (Trend-Following) Strategy

A momentum strategy bets that **price movements persist**: if the market went up
last period, it is more likely to keep going up, and vice-versa. We *follow* the
trend rather than fighting it.

This notebook builds the whole idea from scratch with nothing but pandas — no
machine learning, no heavy math. The steps are:

1. Turn prices into **log returns**.
2. Use the **previous** period's return as our only signal.
3. Check whether a tradable pattern actually exists (in-sample **and**
   out-of-sample).
4. Backtest the rule, measure it, and subtract realistic **trading fees**.

> This is the exact mirror of the [mean reversion](02-mean_reversion.ipynb)
> notebook — the *only* difference is the sign of the signal.

## The data

Weekly (`1w`) OHLC bars for one asset. The columns we care about:

| col | meaning |
|-----|---------|
| `t` | timestamp (bar open) |
| `o` `h` `l` `c` | open / high / low / close price |
| `v` `n` | volume / number of trades |

We only need the **close** (`c`); everything downstream is derived from it.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

# The sample OHLC data ships with the repo under data/samples/. Resolve it
# whether the notebook is launched from the repo root (VS Code default) or from
# its own folder, so "Run All" works either way.
csv_name = 'momentum_ohlc.csv'
csv_path = next(p for p in (Path('data/samples') / csv_name,
                            Path('../../data/samples') / csv_name) if p.exists())

df = pd.read_csv(csv_path)
df = df.set_index('t')
df

## Step 1 — Log returns and the lagged signal

A **log return** is `ln(price_t / price_{t-1})`. We use logs (instead of simple
percentage changes) because they **add up over time**: the sum of the log
returns is the total compound return, which makes the equity-curve math trivial
later.

The **signal** is the *previous* bar's return (`lag_1`). That is the only piece
of information the strategy is allowed to use to decide today's trade — it never
peeks at the current bar.

In [ ]:
# Log return of the close, and the same series shifted one bar into the past.
df['close_log_return'] = np.log(df['c'] / df['c'].shift())
df['close_log_return_lag_1'] = df['close_log_return'].shift()
df

We reduce the lagged return to just its **direction** with `sign`:
`+1` if last bar closed up, `-1` if it closed down. This is what we will bet on.

In [ ]:
# Direction of the previous bar: +1 = up, -1 = down.
df['close_log_return_dir_lag_1'] = np.sign(df['close_log_return_lag_1'])
df

## Step 2 — Is there a tradable pattern?

Before trading anything, we check whether the previous return says *anything*
about the next one. Two quick lenses:

- **Correlation** between today's return and yesterday's. Positive ⇒ momentum.
- **Grouping** today's return by yesterday's *direction*. If the "up" group has a
  positive average and the "down" group a negative one, prior direction tends to
  repeat — that is momentum.

In [ ]:
df[['close_log_return', 'close_log_return_lag_1']].corr()

In [ ]:
# Average / count / total of today's return, grouped by yesterday's direction.
df.groupby('close_log_return_dir_lag_1').aggregate({'close_log_return': ['sum', 'mean', 'count']})

**Reading it:** if the `+1` row has a positive mean and the `-1` row a
negative mean, the market keeps moving in the same direction — the momentum
effect we want to trade.

## Step 3 — Does it survive out-of-sample?

A pattern that only shows up on the data we looked at is worthless. We split the
series chronologically: the first **75%** (in-sample) is where we'd "discover"
the effect, the last **25%** (out-of-sample) is untouched data that stands in for
the future. The pattern must hold in **both**.

In [ ]:
i = int(len(df) * 0.75)
in_sample, out_sample = df.iloc[:i], df.iloc[i:]

In [ ]:
# In-sample: the effect we would have found while researching.
in_sample.groupby('close_log_return_dir_lag_1').aggregate({'close_log_return': ['sum', 'mean', 'count']})

In [ ]:
# Out-of-sample: the real test — data the "research" never saw.
out_sample.groupby('close_log_return_dir_lag_1').aggregate({'close_log_return': ['sum', 'mean', 'count']})

If both tables tell the same story, the momentum behaviour is stable over
time and worth backtesting.

## Step 4 — Backtest the rule

The rule is deliberately simple: **trade in the same direction as the last bar.**

- `signal = +1` → go long (bet up); `signal = -1` → go short (bet down).
- The return we actually earn on a bar is `signal * close_log_return`. When the
  signal matches the market's move, the product is positive; when it's wrong,
  it's negative.
- Summing those trade returns (in log space) gives the **equity curve**.

In [ ]:
# Momentum: bet in the SAME direction as the previous bar.
df['signal'] = df['close_log_return_dir_lag_1']

In [ ]:
# What we earn per bar: our direction times the market's actual move.
df['trade_log_return'] = df['close_log_return'] * df['signal']
df

In [ ]:
# Cumulative log return = the strategy's equity curve (before fees).
df['cum_trade_log_return'] = df['trade_log_return'].cumsum()
df['cum_trade_log_return'].plot(title='Equity curve (gross, log returns)')

## Step 5 — How good is it?

- **Win rate** — fraction of bars that made money.
- **Mean / std** of the per-bar return — the raw edge and its volatility.
- **Sharpe** — edge per unit of risk, annualized.

In [ ]:
df['is_won'] = df['trade_log_return'] > 0
df['is_won'].mean()   # win rate

In [ ]:
df['trade_log_return'].mean()   # average return per bar

In [ ]:
df['trade_log_return'].std()    # volatility per bar

### Annualized Sharpe

Sharpe = mean / std of the per-bar return, scaled to a year. This dataset uses
**weekly (`1w`)** bars, so there are `365 / 7 ≈ 52` periods per year and we scale
by `sqrt(52)`.

In [ ]:
df['trade_log_return'].mean() / df['trade_log_return'].std() * np.sqrt(365 / 7)

## Step 6 — Subtract trading fees

Every bar we re-enter, so each bar is a full **round trip** (an entry + an exit),
and each side pays an exchange fee. Fees are quoted in **basis points**
(1 bp = 0.01%). Exchanges charge two rates:

- **taker** — you cross the book with a market order (higher fee).
- **maker** — you post a limit order and wait (lower fee).

This strategy trades on the close, so we assume the aggressive **taker** rate on
both sides. We track the account value in dollars to apply the fee to the actual
notional traded.

In [ ]:
CAPITAL = 1000

# Notional value of the account after each bar's gross P&L (before fees).
df['post_trade_notional_value'] = CAPITAL + CAPITAL * df['cum_trade_log_return']
# Value going into the bar = previous bar's post value (seed the first with CAPITAL).
df['pre_trade_notional_value'] = df['post_trade_notional_value'].shift().fillna(CAPITAL)
df

In [ ]:
TAKER_FEE_BPS = 4.1   # aggressive (market order)
MAKER_FEE_BPS = 1.2   # passive   (limit order) — shown for reference

TAKER_FEE_PCT = TAKER_FEE_BPS / 10_000
MAKER_FEE_PCT = MAKER_FEE_BPS / 10_000

# Round-trip taker fee: pay on the way in (on pre-trade notional) and out (post-trade).
df['entry_fee'] = df['pre_trade_notional_value'] * TAKER_FEE_PCT
df['exit_fee'] = df['post_trade_notional_value'] * TAKER_FEE_PCT
df['tx_fees'] = df['entry_fee'] + df['exit_fee']
df['cum_tx_fees'] = df['tx_fees'].cumsum()
df

In [ ]:
# Net equity = gross account value minus the fees paid so far.
df['net_equity'] = df['post_trade_notional_value'] - df['cum_tx_fees']
df

### Gross vs net

Plotting both side by side shows how much of the edge the fees eat. For a
strategy that trades every single bar, transaction costs are often the difference
between a profitable and a losing system.

In [ ]:
ax = df['post_trade_notional_value'].plot(label='gross', legend=True)
df['net_equity'].plot(ax=ax, label='net (after fees)', legend=True, title='Equity: gross vs net ($)')

## Exercises

1. **Position sizing** — instead of a constant $1000, compound the account so
   each bar risks the *current* equity.
2. **Fee sensitivity** — how high can the taker fee go before the edge vanishes?
3. **Signal depth** — does using the last *two* bars' directions beat using one?
4. **Compare** — run the [mean reversion](02-mean_reversion.ipynb) notebook on the
   same asset. Which regime does this market actually favour?